# Retrain the EcBiz 0264 marker — Google Colab (single T4)

1. Runtime → Change runtime type → **T4 GPU**.
2. Run cell 1 and choose `kaggle-training-pack.zip` from your computer.
3. Run the remaining cells in order. No Kaggle account or queue needed.
4. Add `HF_TOKEN` in Colab Secrets (🔑) with a Hugging Face **write** token, or type it when asked.

In [ ]:
from google.colab import files
import zipfile, pathlib, os
ROOT = pathlib.Path('/content/training')
ROOT.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()  # choose kaggle-training-pack.zip
zip_path = next(iter(uploaded))
with zipfile.ZipFile(zip_path) as z:
    z.extractall(ROOT)
print('files:', sorted(p.name for p in ROOT.iterdir()))
assert (ROOT / 'dataset' / 'train.jsonl').exists(), 'dataset missing from zip'

In [ ]:
!pip -q uninstall -y diffusers gradio gradio-client optimum
!pip -q install -U "transformers>=4.44,<5" peft accelerate sentencepiece "optimum>=2.1,<3" "optimum-onnx>=0.1" onnx onnxruntime-gpu huggingface_hub

Train. The script defaults are already T4-safe: batch 2, accumulation 16, sequence 768, expandable CUDA segments.

In [ ]:
!cd /content/training && python kaggle_train.py

In [ ]:
!cd /content/training && python export_onnx.py

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF token loaded from Colab Secrets')
except Exception:
    import getpass
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face write token: ')
!cd /content/training && python upload_to_hf.py